In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd


# ============================================================
# PATH
# ============================================================

PRIORITY_FILES = [

    Path(
        r"..\Data Kementrian Koordinator dan FKTP"
        r"\DesaPrioritas1_MALUKU_filter_faskes_2026-07-18.xlsx"
    ),

    Path(
        r"..\Data Kementrian Koordinator dan FKTP"
        r"\DesaPrioritas1_MALUKU_UTARA_filter_faskes_2026-07-18.xlsx"
    )

]


ZIP_FILE = Path(

    r".."
    r"\BATAS WILAYAH KELURAHAN-DESA 10K.zip"

)


SHP = (

    f"/vsizip/{ZIP_FILE.as_posix()}"

    "/Batas_Wilayah_KelurahanDesa_10K_AR.shp"

)


# ============================================================
# NORMALIZATION FUNCTIONS
# ============================================================

def normalize_code(series):

    return (

        series

        .astype(str)

        .str.replace(r"\D", "", regex=True)

        .str.strip()

    )


def normalize_text(series):

    return (

        series

        .astype(str)

        .str.upper()

        .str.strip()

        .str.replace(

            r"\s+",

            " ",

            regex=True

        )

    )


# ============================================================
# READ PRIORITY DATA
# ============================================================

priority_list = []


for file in PRIORITY_FILES:

    df = pd.read_excel(file)

    priority_list.append(df)


priority = pd.concat(

    priority_list,

    ignore_index=True

)


print(

    f"Total desa prioritas: {len(priority)}"

)


# ============================================================
# READ BIG SHAPEFILE
# ============================================================

print()

print(

    "Membaca shapefile BIG..."

)


polygons = gpd.read_file(

    SHP

)


polygons = polygons[

    polygons["WADMPR"].isin(

        [

            "Maluku",

            "Maluku Utara"

        ]

    )

].copy()


print(

    f"Total polygon Maluku + Maluku Utara: "

    f"{len(polygons)}"

)


# ============================================================
# NORMALIZE CODE
# ============================================================

priority["_CODE"] = normalize_code(

    priority["Kode Desa"]

)


polygons["_CODE"] = normalize_code(

    polygons["KDEPUM"]

)


# ============================================================
# MATCH BY CODE
# ============================================================

matched_codes = set(

    polygons["_CODE"]

)


unmatched = priority[

    ~priority["_CODE"].isin(

        matched_codes

    )

].copy()


print()

print(

    f"Desa tidak cocok berdasarkan kode: "

    f"{len(unmatched)}"

)


# ============================================================
# NORMALIZE NAMES
# ============================================================

unmatched["_VILLAGE_NAME"] = normalize_text(

    unmatched["Desa/Kelurahan"]

)


unmatched["_DISTRICT"] = normalize_text(

    unmatched["Kecamatan"]

)


unmatched["_REGENCY"] = normalize_text(

    unmatched["Kabupaten/Kota"]

)


unmatched["_PROVINCE"] = normalize_text(

    unmatched["Provinsi"]

)


polygons["_VILLAGE_NAME"] = normalize_text(

    polygons["WADMKD"]

)


polygons["_DISTRICT"] = normalize_text(

    polygons["WADMKC"]

)


polygons["_REGENCY"] = normalize_text(

    polygons["WADMKK"]

)


polygons["_PROVINCE"] = normalize_text(

    polygons["WADMPR"]

)


# ============================================================
# SEARCH BY NAME + LOCATION
# ============================================================

print()

print("=" * 70)

print(

    "SEARCHING UNMATCHED VILLAGES BY NAME"

)

print("=" * 70)


for _, village in unmatched.iterrows():

    print()

    print("-" * 70)


    print(

        "Priority village:"

    )


    print(

        f"Nama       : "

        f"{village['Desa/Kelurahan']}"

    )


    print(

        f"Kecamatan  : "

        f"{village['Kecamatan']}"

    )


    print(

        f"Kabupaten  : "

        f"{village['Kabupaten/Kota']}"

    )


    print(

        f"Kode       : "

        f"{village['Kode Desa']}"

    )


    # --------------------------------------------------------
    # EXACT NAME + ADMINISTRATIVE LOCATION
    # --------------------------------------------------------

    matches = polygons[

        (

            polygons["_VILLAGE_NAME"]

            == village["_VILLAGE_NAME"]

        )

        &

        (

            polygons["_DISTRICT"]

            == village["_DISTRICT"]

        )

        &

        (

            polygons["_REGENCY"]

            == village["_REGENCY"]

        )

        &

        (

            polygons["_PROVINCE"]

            == village["_PROVINCE"]

        )

    ]


    if len(matches) > 0:

        print()

        print(

            "MATCH DITEMUKAN!"

        )


        print(

            matches[

                [

                    "KDEPUM",

                    "WADMKD",

                    "WADMKC",

                    "WADMKK",

                    "WADMPR"

                ]

            ].to_string(

                index=False

            )

        )


    else:

        print()

        print(

            "Tidak ditemukan exact match."

        )


        # ----------------------------------------------------
        # SEARCH NAME ONLY
        # ----------------------------------------------------

        name_matches = polygons[

            polygons["_VILLAGE_NAME"]

            == village["_VILLAGE_NAME"]

        ]


        if len(name_matches) > 0:

            print()

            print(

                "Kemungkinan match berdasarkan nama:"

            )


            print(

                name_matches[

                    [

                        "KDEPUM",

                        "WADMKD",

                        "WADMKC",

                        "WADMKK",

                        "WADMPR"

                    ]

                ].to_string(

                    index=False

                )

            )

        else:

            print()

            print(

                "Nama desa juga tidak ditemukan."

            )

In [ ]:
import pandas as pd
import geopandas as gpd
from difflib import SequenceMatcher


def normalize_text(text):

    return (

        str(text)

        .upper()

        .strip()

        .replace("KAB. ", "")

    )


def similarity(a, b):

    return (

        SequenceMatcher(

            None,

            normalize_text(a),

            normalize_text(b)

        ).ratio()

        * 100

    )


# Priority data
priority_villages = pd.DataFrame({

    "Priority_Name": [

        "URUNG",

        "KWAWOR KECIL MATA ATA",

        "KWAWOR BESAR ENA"

    ],

    "Priority_District": [

        "UKAR SENGAN",

        "UKAR SENGAN",

        "UKAR SENGAN"

    ],

    "Priority_Regency": [

        "KAB. SERAM BAGIAN TIMUR",

        "KAB. SERAM BAGIAN TIMUR",

        "KAB. SERAM BAGIAN TIMUR"

    ],

    "Priority_Province": [

        "MALUKU",

        "MALUKU",

        "MALUKU"

    ]

})


# BIG shapefile matches
big_villages = pd.DataFrame({

    "BIG_Name": [

        "Urung",

        "Kwawor Kecil Mata Ata",

        "Kwawor Besar Ena"

    ],

    "BIG_District": [

        "Seram Timur",

        "Seram Timur",

        "Seram Timur"

    ],

    "BIG_Regency": [

        "Seram Bagian Timur",

        "Seram Bagian Timur",

        "Seram Bagian Timur"

    ],

    "BIG_Province": [

        "Maluku",

        "Maluku",

        "Maluku"

    ]

})


# Calculate similarity
result = priority_villages.copy()


result["Name_Similarity_%"] = [

    similarity(

        priority_villages.loc[i, "Priority_Name"],

        big_villages.loc[i, "BIG_Name"]

    )

    for i in range(len(priority_villages))

]


result["Regency_Same"] = [

    normalize_text(

        priority_villages.loc[i, "Priority_Regency"]

    )

    ==

    normalize_text(

        big_villages.loc[i, "BIG_Regency"]

    )

    for i in range(len(priority_villages))

]


result["Province_Same"] = [

    normalize_text(

        priority_villages.loc[i, "Priority_Province"]

    )

    ==

    normalize_text(

        big_villages.loc[i, "BIG_Province"]

    )

    for i in range(len(priority_villages))

]


result["District_Same"] = [

    normalize_text(

        priority_villages.loc[i, "Priority_District"]

    )

    ==

    normalize_text(

        big_villages.loc[i, "BIG_District"]

    )

    for i in range(len(priority_villages))

]


result